In [5]:
import pandas as pd 
import networkx as nx 
import numpy as np 
from sklearn.metrics import roc_auc_score

In [3]:
edgelist = pd.read_csv("twitch_edgelist_10K.csv")

nodes = pd.read_csv("twitch_nodes_10k_final.csv")

In [7]:
edgelist.head(3)

,Source,Target,Type,Weight
0,273316640,207277102,Directed,1
1,61294188,17182295,Directed,1
2,55743821,108547363,Directed,1


In [8]:
nodes.head(3)

,ID,Label,Broadcaster_Type,Total_Followers
0,77827128,Tumblurr,partner,2006052
1,539141014,aNcMedia,partner,4582
2,179279586,Kertus,partner,170260


In [ ]:
G = nx.from_pandas_edgelist(
    edgelist,
    source='Source',
    target='Target',
    create_using=nx.DiGraph() 
)

node_attributes = nodes.set_index('ID').to_dict(orient='index') #creo un dizionario dove la key è l'ID ed i values sono gli attributi del nodo 
nx.set_node_attributes(G, node_attributes) # aggiungo gli attributi ai nodi del grafo 

In [13]:
print(f"Grafo Orientato: {nx.is_directed(G)}")
print(f"Numero totale di nodi (streamer): {G.number_of_nodes()}")
print(f"Numero totale di archi (interazioni): {G.number_of_edges()}")

Grafo Orientato: True
Numero totale di nodi (streamer): 10355
Numero totale di archi (interazioni): 17433


In [ ]:


# 1. Carica la edgelist e crea il grafo non orientato
edgelist = pd.read_csv('twitch_edgelist_10k.csv')
G_full = nx.from_pandas_edgelist(edgelist, source='Source', target='Target')

# 2. Dividi gli archi in 80% Train e 20% Test
all_edges = list(G_full.edges())
np.random.shuffle(all_edges) # Mescola gli archi casualmente

split_point = int(0.8 * len(all_edges))
train_edges = all_edges[:split_point]
test_pos_edges = all_edges[split_point:]

# 3. Crea il grafo di allenamento (con l'80% degli archi)
G_train = nx.Graph()
G_train.add_nodes_from(G_full.nodes())
G_train.add_edges_from(train_edges)

# 4. Genera archi negativi casuali (archi che NON esistono nel grafo)
nodes = list(G_full.nodes())
test_neg_edges = []
while len(test_neg_edges) < len(test_pos_edges):
    u, v = np.random.choice(nodes, size=2, replace=False)
    if not G_full.has_edge(u, v) and (u, v) not in test_neg_edges:
        test_neg_edges.append((u, v))

# Uniamo i test positivi e negativi in un'unica lista e creiamo le etichette (1=esiste, 0=non esiste)
test_samples = test_pos_edges + test_neg_edges
y_true = [1] * len(test_pos_edges) + [0] * len(test_neg_edges)

# 5. Applica gli algoritmi classici di NetworkX
# Common Neighbors (Vicini Comuni)
score_cn = [len(list(nx.common_neighbors(G_train, u, v))) for u, v in test_samples]

# Jaccard Coefficient
score_jc = [p for u, v, p in nx.jaccard_coefficient(G_train, test_samples)]

# Adamic-Adar Index
score_aa = [p for u, v, p in nx.adamic_adar_index(G_train, test_samples)]

# Preferential Attachment
score_pa = [p for u, v, p in nx.preferential_attachment(G_train, test_samples)]

# 6. Calcola e stampa l'accuratezza (ROC-AUC)
print("--- ACCURATEZZA (ROC-AUC) ---")
print(f"Common Neighbors:        {roc_auc_score(y_true, score_cn):.4f}")
print(f"Jaccard Coefficient:     {roc_auc_score(y_true, score_jc):.4f}")
print(f"Adamic Adar:             {roc_auc_score(y_true, score_aa):.4f}")
print(f"Preferential Attachment: {roc_auc_score(y_true, score_pa):.4f}")



--- ACCURATEZZA (ROC-AUC) ---
Common Neighbors:        0.5278
Jaccard Coefficient:     0.4442
Adamic Adar:             0.5283
Preferential Attachment: 0.4149
